<a href="https://colab.research.google.com/github/thetallguy14/flyrank-ml-internship/blob/main/Copy_of_w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
## Baseline Rule

The goal of this baseline rule is to identify pages that are good candidates for optimization using observable search performance metrics.

Each page receives a baseline score based on three signals:

- Lower Click-Through Rate (CTR) → higher priority because the page may benefit from improved titles or meta descriptions.
- Worse Average Search Position → higher priority because improving ranking could increase visibility.
- Higher Search Impressions → higher priority because optimizing these pages has greater potential impact.

The baseline score combines these three signals into a single ranking score. Pages with the highest scores are placed at the top of the action queue for review.

### Action Label

REFRESH_CONTENT

### Reason Codes

- LOW_CTR
- LOW_RANKING
- HIGH_IMPRESSIONS

Each page receives the reason code that best explains why it was prioritized.

In [ ]:
work_df = df.copy()

In [ ]:
print(work_df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import numpy as np
import os

# Work on a copy
work_df = df.copy()

# -----------------------------
# Create CTR
# -----------------------------
work_df["ctr"] = (
    work_df["gsc_clicks"] /
    work_df["gsc_impressions"].replace(0, np.nan)
)

work_df["ctr"] = work_df["ctr"].fillna(0)

# -----------------------------
# Normalize signals
# -----------------------------

# Lower CTR = higher priority
work_df["ctr_score"] = 1 - work_df["ctr"]

# Worse search position = higher priority
work_df["position_score"] = (
    work_df["gsc_avg_position"] /
    work_df["gsc_avg_position"].max()
)

# Higher impressions = higher priority
work_df["impression_score"] = (
    work_df["gsc_impressions"] /
    work_df["gsc_impressions"].max()
)

# -----------------------------
# Baseline Score
# -----------------------------
work_df["baseline_score"] = (
    0.50 * work_df["ctr_score"] +
    0.30 * work_df["position_score"] +
    0.20 * work_df["impression_score"]
)

# -----------------------------
# Reason Code
# -----------------------------
def get_reason(row):

    if row["ctr"] < 0.03:
        return "LOW_CTR"

    elif row["gsc_avg_position"] > 20:
        return "LOW_RANKING"

    else:
        return "HIGH_IMPRESSIONS"

work_df["reason_code"] = work_df.apply(get_reason, axis=1)

# -----------------------------
# Action Label
# -----------------------------
work_df["action"] = "REFRESH_CONTENT"

# -----------------------------
# Rank Pages
# -----------------------------
ranked_queue = (
    work_df
    .sort_values("baseline_score", ascending=False)
    .reset_index(drop=True)
)

# -----------------------------
# Save CSV
# -----------------------------
os.makedirs("work/outputs", exist_ok=True)

ranked_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully!")

# Preview
ranked_queue[
    [
        "content_hash_id",
        "baseline_score",
        "action",
        "reason_code"
    ]
].head(10)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
## Top-20 Review

The top-ranked pages were reviewed using the baseline score, reason code, and search metrics.

Most high-ranked pages showed one or more of the following:

- Low click-through rate
- Poor average search position
- High search impressions

Confidence is moderate because the rule uses only current observable metrics and does not evaluate page quality or search intent.

A recommendation could be incorrect if:

- the page was recently updated,
- search behaviour is seasonal,
- tracking data is incomplete,
- or low CTR is expected for that type of query.

The ranked queue should therefore be treated as decision support rather than a final recommendation.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
## Weak Picks

Some pages may receive a high score even though they are not the best optimization candidates.

Possible weak picks include:

- Pages with very low traffic where CTR is unstable.
- Pages affected by seasonal search trends.
- Pages with incomplete analytics data.
- Pages that have already been optimized but recent improvements are not yet reflected.

## Leakage Check

No future information or product-generated flags were used.

The baseline score only uses current observable metrics:

- Google Search impressions
- Google Search clicks
- Average search position

No future outcomes, labels, or target variables were included when calculating the score, so the rule does not suffer from data leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.